![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-4-h-small` to perform chat conversation with JSON response format

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook provides a detailed demonstration of the steps and code required to showcase support for JSON response format in Chat models.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The purpose of this notebook is to demonstrate how to use JSON response format in Chat models and how to specify the response JSON schema.

## Table of Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Set up the Foundation Model on IBM watsonx.ai](#Set-up-the-Foundation-Model-on-IBM-watsonx.ai)
3. [Work with JSON response format](#Work-with-JSON-response-format)
4. [Work with specified JSON schema response format](#Work-with-specified-JSON-schema-response-format)
5. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install dependencies

**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U "ibm-watsonx-ai" | tail -n 1

### Define the watsonx.ai credentials
Use the code cell below to define the watsonx.ai credentials that are required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">Managing user API keys</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

### Define the project ID
You need to provide the project ID to give the Foundation Model the context for the call. If you have a default project ID set in Watson Studio, the notebook obtains that project ID. Otherwise, you need to provide the project ID in the code cell below.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Enter your project_id and hit enter: ")

<a id="Set-up-the-Foundation-Model-on-IBM-watsonx.ai"></a>
## Set up the Foundation Model on IBM watsonx.ai

Specify the `model_id` of the model you will use for the chat with tools.

In [4]:
model_id = "ibm/granite-4-h-small"

### Model parameters overview

In order to receive the response from the model in the JSON format, the `TextChatResponseFormatType.JSON_OBJECT` response format must be specified. You might also need to adjust model parameters depending on the model you use.

In [5]:
from ibm_watsonx_ai.foundation_models.schema import TextChatParameters

TextChatParameters.show()

+-----------------------+----------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| PARAMETER             | TYPE                                   | EXAMPLE VALUE                                                                                                                                                                                                                                                                   |
+=======================+========================================+============================================================================================================================================================================================================================================================

In [6]:
from ibm_watsonx_ai.foundation_models.schema import (
    TextChatResponseFormat,
    TextChatResponseFormatType,
)

TextChatResponseFormat.show()

+-------------+--------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| PARAMETER   | TYPE                                       | EXAMPLE VALUE                                                                                                                                                                                                                           |
+=============+============================================+=========================================================================================================================================================================================================================================+
| type        | str, TextChatResponseFormatType            | json_schema                                           

<a id="Work-with-JSON-response-format"></a>
## Work with JSON response format

### Initialize the model

Initialize the `ModelInference` class with provided parameters.

In [7]:
from ibm_watsonx_ai.foundation_models import ModelInference

params = TextChatParameters(
    response_format=TextChatResponseFormat(TextChatResponseFormatType.JSON_OBJECT),
    max_tokens=1024,
    temperature=1,
)

model = ModelInference(
    model_id=model_id, credentials=credentials, project_id=project_id, params=params
)

### Create messages and chat with the model

In order to ensure the response is in the correct format, the sent messages must contain an indication that JSON is expected.

In [8]:
messages = [
    {"role": "system", "content": "Respond in a JSON format"},
    {"role": "user", "content": "Describe methods of calculating pi"},
]

chat_response = model.chat(messages=messages, params=params)

### Parse the response

Use the `json` library to parse the chat response content into a Python-native data structure.

In [9]:
import json

json_response_content = json.loads(chat_response["choices"][0]["message"]["content"])
print(json.dumps(json_response_content, ensure_ascii=False, indent=2))

{
  "methods": [
    {
      "name": "Leibniz formula",
      "description": "An infinite series that converges to pi. It is given by the formula: pi/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 - ..."
    },
    {
      "name": "Nilakantha series",
      "description": "An infinite series developed by the Indian mathematician Nilakantha Somayaji. It converges to pi faster than the Leibniz formula. The series is: pi = 3 + 4/(2*3*4) - 4/(4*5*6) + 4/(6*7*8) - 4/(8*9*10) + ..."
    },
    {
      "name": "Monte Carlo method",
      "description": "A probabilistic approach to estimate pi. It involves randomly placing points inside a square and calculating the ratio of points inside the inscribed circle to the total points. This ratio, multiplied by 4, approximates pi."
    },
    {
      "name": "Chudnovsky algorithm",
      "description": "A fast converging series that is currently one of the most efficient methods for calculating pi to a large number of decimal places. It is based on a formula involvin

<a id="Work-with-specified-JSON-schema-response-format"></a>
## Work with specified JSON schema response format

### Initialize the model for JSON schema response

Initialize a new `ModelInference` class with the provided parameters, including the expected JSON schema. For more info about JSON schema, visit: https://json-schema.org/learn

In [10]:
params = TextChatParameters(
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "Cake recipes",
            "schema": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "cake_name": {
                            "type": "string",
                        },
                        "description": {
                            "type": "string",
                        },
                        "difficulty_score": {
                            "type": "integer",
                        },
                        "expected_price": {
                            "type": "number",
                        },
                    },
                },
            },
            "strict": True,
        },
    }
)

model = ModelInference(
    model_id=model_id, credentials=credentials, project_id=project_id, params=params
)

### Create messages using JSON schema and chat with the model

As previously, in order to ensure the response is in the correct format, the sent message must contain an indication that JSON is expected.

In [11]:
messages = [
    {
        "role": "system",
        "content": "Respond in a JSON format.",
    },
    {
        "role": "user",
        "content": "Provide a list of cake recipes. Briefly describe what the cake tastes like. Give each a difficulty score between 1-10. Also add an expected price of the cake with accuracy of 2 decimal places.",
    },
]

chat_response = model.chat(messages=messages)

### Parse the response with JSON schema

As previously, use the `json` library to parse the chat response content into a Python-native data structure.

In [12]:
json_response_content = json.loads(chat_response["choices"][0]["message"]["content"])
print(json.dumps(json_response_content, ensure_ascii=False, indent=2))

[
  {
    "cake_name": "Vanilla Cake",
    "description": "A classic, moist, and fluffy vanilla cake with a sweet and creamy taste.",
    "difficulty_score": 3,
    "expected_price": 25.0
  },
  {
    "cake_name": "Chocolate Cake",
    "description": "A rich, decadent, and moist chocolate cake that is a crowd-pleaser.",
    "difficulty_score": 4,
    "expected_price": 30.0
  },
  {
    "cake_name": "Red Velvet Cake",
    "description": "A velvety, slightly tart red velvet cake with a luscious cream cheese frosting.",
    "difficulty_score": 5,
    "expected_price": 35.0
  },
  {
    "cake_name": "Lemon Drizzle Cake",
    "description": "A zesty and moist lemon cake with a sweet lemon drizzle glaze on top.",
    "difficulty_score": 3,
    "expected_price": 28.0
  },
  {
    "cake_name": "Carrot Cake",
    "description": "A sweet and moist carrot cake with warm spices and a creamy frosting.",
    "difficulty_score": 4,
    "expected_price": 32.0
  },
  {
    "cake_name": "Black Forest Ca

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to work with chat models using tools and watsonx.ai.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.